<a href="https://colab.research.google.com/github/beyzaturku/2209/blob/main/SRCNN/ResNet_SRCNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
from google.colab import drive

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In [151]:
from PIL import Image
import os
import numpy as np
import math
import cv2
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, Add, LeakyReLU, Activation, Lambda, Dropout, Layer, GlobalAveragePooling2D, Dense
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.optimizers import SGD, Adam
from tensorflow.keras.regularizers import l2
import tensorflow as tf
from tensorflow import keras
from IPython import get_ipython
from IPython.display import display
import torch

In [172]:
# Veri augmentation ile patch generator
def generate_patches(hr_dir, lr_dir, patch_size=32, scale=2, batch_size=8):
    hr_files = sorted(os.listdir(hr_dir))
    lr_files = sorted(os.listdir(lr_dir))

    while True:
        lr_batch = []
        hr_batch = []
        for _ in range(batch_size):
            idx = np.random.randint(0, len(hr_files))
            hr_path = os.path.join(hr_dir, hr_files[idx])
            lr_path = os.path.join(lr_dir, lr_files[idx])

            hr_img = Image.open(hr_path).convert('L')
            hr_img = np.array(hr_img, dtype=np.float32) / 255.0

            lr_img = Image.open(lr_path).convert('L')
            lr_img = np.array(lr_img, dtype=np.float32) / 255.0

            # LR görüntü için uygun bir patch seçin
            lr_h, lr_w = lr_img.shape
            lr_y = np.random.randint(0, lr_h - patch_size + 1)
            lr_x = np.random.randint(0, lr_w - patch_size + 1)
            lr_patch = lr_img[lr_y:lr_y+patch_size, lr_x:lr_x+patch_size]

            # HR görüntü için uygun boyutta patch seçin
            hr_h, hr_w = hr_img.shape
            hr_patch_size = patch_size * scale
            hr_y = lr_y * scale
            hr_x = lr_x * scale

            # Sınırları kontrol edin
            if hr_y + hr_patch_size > hr_h or hr_x + hr_patch_size > hr_w:
                continue

            hr_patch = hr_img[hr_y:hr_y+hr_patch_size, hr_x:hr_x+hr_patch_size]

            # Data augmentation (aynı)
            if np.random.random() > 0.5:
                brightness = np.random.uniform(0.8, 1.2)
                lr_patch *= brightness
                hr_patch *= brightness
                lr_patch = np.clip(lr_patch, 0, 1)
                hr_patch = np.clip(hr_patch, 0, 1)

            lr_batch.append(lr_patch)
            hr_batch.append(hr_patch)

        lr_batch = np.expand_dims(np.array(lr_batch), -1)  # (batch_size, 32, 32, 1)
        hr_batch = np.expand_dims(np.array(hr_batch), -1)  # (batch_size, 64, 64, 1)
        yield lr_batch, hr_batch

In [170]:
def psnr_loss(y_true, y_pred):
    max_val = 1.0
    return -tf.math.reduce_mean(tf.image.psnr(y_true, y_pred, max_val))

def psnr_metric(y_true, y_pred):
    return tf.reduce.mean(tf.image.psnr(y_true, y_pred, max_val=1.0))

In [166]:
# Residual Blok
def res_block(x, filters=64, kernel_size=(3, 3)):
    skip = x
    x = Conv2D(filters, kernel_size, padding='same', activation='relu')(x)
    x = Conv2D(filters, kernel_size, padding='same')(x)
    x = Dropout(0.1)(x)
    x = Add()([x, skip])
    return x

# Sub-Pixel Convolution Katmanı
class SubPixelConv2D(Layer):
    def __init__(self, upscale_factor=2, **kwargs):
        super(SubPixelConv2D, self).__init__(**kwargs)
        self.upscale_factor = upscale_factor

    def build(self, input_shape):
        self.channels = input_shape[-1] // (self.upscale_factor ** 2)
        super(SubPixelConv2D, self).build(input_shape)

    def call(self, inputs):
        return tf.nn.depth_to_space(inputs, self.upscale_factor)

    def compute_output_shape(self, input_shape):
        batch, height, width, channels = input_shape
        new_height = height * self.upscale_factor if height else None
        new_width = width * self.upscale_factor if width else None
        new_channels = channels // (self.upscale_factor ** 2)
        return (batch, new_height, new_width, new_channels)

    def get_config(self):
        config = super(SubPixelConv2D, self).get_config()
        config.update({'upscale_factor': self.upscale_factor})
        return config

# Upscale Bloğu
def upscale_block(x, filters=64):
    x = Conv2D(filters * 4, (3, 3), padding='same', activation='relu')(x)
    x = SubPixelConv2D(upscale_factor=2)(x)
    return x

# Revize edilmiş ResNet-SRCNN
def create_resnet_srcnn(input_shape=(None, None, 1)):
    inputs = Input(shape=input_shape)
    x = inputs

    x = Conv2D(64, (9, 9), padding='same', activation='relu')(x) #İlk özellik çıkarma

    for _ in range(10): #Residual bloklar
        x = res_block(x, filters=64)

    x = Conv2D(32, (5, 5), padding='same', activation='relu')(x)
    x = upscale_block(x)
    x = Conv2D(1, (3, 3), padding='same')(x)

    # Normalizasyon geri dönüşümünü kaldırıyoruz
    # outputs = Lambda(lambda x: x * 255.0)(x)
    outputs = x

    model = Model(inputs=inputs, outputs=outputs)
    optimizer = Adam(learning_rate=0.0001, weight_decay=0.0001)

    # Özel bir PSNR metriği tanımlayalım
    def psnr_metric(y_true, y_pred):
        return tf.reduce_mean(tf.image.psnr(y_true, y_pred, max_val=1.0))

    model.compile(
        optimizer=optimizer,
        loss=psnr_loss,
        metrics=[psnr_metric]
    )

    return model

In [167]:
model.summary()

Model: "functional_18"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_15            │ (None, 32, 32, 1)      │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_248 (Conv2D)       │ (None, 32, 32, 64)     │          5,248 │ input_layer_15[0][0]   │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_249 (Conv2D)       │ (None, 32, 32, 64)     │         36,928 │ conv2d_248[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_250 (Conv2D)       │ (None, 32, 32, 64)     │         36,928 │ conv2d_249[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_70 (Dropout)      │ (None, 32, 32, 64)     │              0 │ conv2d_250[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_100 (Add)             │ (None, 32, 32, 64)     │              0 │ dropout_70[0][0],      │
│                           │                        │                │ conv2d_248[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_251 (Conv2D)       │ (None, 32, 32, 64)     │         36,928 │ add_100[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_252 (Conv2D)       │ (None, 32, 32, 64)     │         36,928 │ conv2d_251[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_71 (Dropout)      │ (None, 32, 32, 64)     │              0 │ conv2d_252[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_101 (Add)             │ (None, 32, 32, 64)     │              0 │ dropout_71[0][0],      │
│                           │                        │                │ add_100[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_253 (Conv2D)       │ (None, 32, 32, 64)     │         36,928 │ add_101[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_254 (Conv2D)       │ (None, 32, 32, 64)     │         36,928 │ conv2d_253[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_72 (Dropout)      │ (None, 32, 32, 64)     │              0 │ conv2d_254[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_102 (Add)             │ (None, 32, 32, 64)     │              0 │ dropout_72[0][0],      │
│                           │                        │                │ add_101[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_255 (Conv2D)       │ (None, 32, 32, 64)     │         36,928 │ add_102[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_256 (Conv2D)       │ (None, 32, 32, 64)     │         36,928 │ conv2d_255[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_73 (Dropout)      │ (None, 32, 32, 64)     │              0 │ conv2d_256[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_103 (Add)        

 Total params: 2,608,805 (9.95 MB)

 Trainable params: 869,601 (3.32 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 1,739,204 (6.63 MB)

In [168]:
if __name__ == "__main__":
    # Klasör yolları
    hr_dir = "/content/drive/MyDrive/srcnn_dataset/dataset_srcnn/HR/train"  # 1024x540 görüntüler
    lr_dir = "/content/drive/MyDrive/srcnn_dataset/dataset_srcnn/LR/train"  # 512x270 görüntüler

    model = create_resnet_srcnn()
    train_generator = generate_patches(hr_dir, lr_dir, patch_size=32, batch_size=8)
    steps_per_epoch = 1654 // 8
    early_stopping = tf.keras.callbacks.EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)

    checkpoint = ModelCheckpoint(
    filepath='best_model.weights.h5',
    monitor='psnr_metric',
    save_best_only=True,
    save_weights_only=True,
    mode='max'
)

In [134]:
# TensorFlow ağırlıklarını PyTorch formatına dönüştürme
def save_weights_to_pt(model, filepath):
    weights = {layer.name: layer.get_weights() for layer in model.layers if layer.get_weights()}
    torch_weights = {}
    for name, weight in weights.items():
        for i, w in enumerate(weight):
            torch_weights[f"{name}_{i}"] = torch.tensor(w)
    torch.save(torch_weights, filepath)

In [173]:
history = model.fit(
        train_generator,
        steps_per_epoch=steps_per_epoch,
        epochs=100,
        callbacks=[early_stopping, checkpoint]
    )

model.save('final2_model.h5')
model.save_weights('final2_weights.weights.h5')
save_weights_to_pt(model, 'final2_weights.pt')

Epoch 1/100


ValueError: Attr 'Toutput_types' of 'OptionalFromValue' Op passed list of length 0 less than minimum 1.

In [150]:
print("Eğitim Sonuçları:")
print("Loss (-PSNR):", history.history['loss'])
print("PSNR:", history.history['psnr_metric'])

Eğitim Sonuçları:
Loss (-PSNR): [-22.45015525817871, -25.30198097229004, -25.90555763244629, -26.227001190185547, -26.597999572753906, -26.76620864868164, -27.073143005371094, -26.975994110107422, -27.529075622558594, -27.010696411132812, -27.257265090942383, -27.863893508911133, -27.694387435913086, -27.363115310668945, -27.65500831604004, -27.502500534057617, -27.90431785583496, -27.84398078918457, -28.04714584350586, -27.451786041259766, -27.736719131469727, -27.759727478027344, -28.295652389526367, -28.035612106323242, -28.201868057250977, -27.938066482543945, -28.210447311401367, -28.216514587402344, -28.14894676208496, -28.26673126220703, -28.246232986450195, -28.184728622436523, -28.303098678588867, -28.352745056152344, -28.481903076171875, -28.143009185791016, -28.345321655273438, -28.22181510925293, -28.291994094848633, -28.663131713867188, -28.35004234313965, -28.50553321838379, -28.570920944213867, -28.803457260131836, -28.417724609375, -28.57232666015625, -28.47475433349609

## Model Test Süreci

In [161]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim
from PIL import Image
import cv2
from tqdm import tqdm
from tensorflow.keras.layers import Input, Conv2D, Add, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

In [157]:
# Özel katmanımızı yükleme sırasında tanımlamamız gerekiyor
class SubPixelConv2D(tf.keras.layers.Layer):
    def __init__(self, upscale_factor=2, **kwargs):
        super(SubPixelConv2D, self).__init__(**kwargs)
        self.upscale_factor = upscale_factor

    def build(self, input_shape):
        self.channels = input_shape[-1] // (self.upscale_factor ** 2)
        super(SubPixelConv2D, self).build(input_shape)

    def call(self, inputs):
        return tf.nn.depth_to_space(inputs, self.upscale_factor)

    def compute_output_shape(self, input_shape):
        batch, height, width, channels = input_shape
        new_height = height * self.upscale_factor if height else None
        new_width = width * self.upscale_factor if width else None
        new_channels = channels // (self.upscale_factor ** 2)
        return (batch, new_height, new_width, new_channels)

    def get_config(self):
        config = super(SubPixelConv2D, self).get_config()
        config.update({'upscale_factor': self.upscale_factor})
        return config

# PSNR kaybı
def psnr_loss(y_true, y_pred):
    max_val = 1.0
    return -tf.reduce_mean(tf.image.psnr(y_true, y_pred, max_val))

# PSNR metriği
def psnr_metric(y_true, y_pred):
    return tf.reduce_mean(tf.image.psnr(y_true, y_pred, max_val=1.0))

In [158]:
# Görüntü yükleme ve ön işleme fonksiyonu
def load_and_preprocess_image(image_path, grayscale=True, normalize=True):
    if grayscale:
        # Gri tonlamalı görüntü yükleme
        img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        img = np.expand_dims(img, axis=-1)  # Kanal boyutu ekleme
    else:
        # Renkli görüntü yükleme
        img = cv2.imread(image_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    if normalize:
        img = img.astype(np.float32) / 255.0

    return img

In [160]:
# Test görüntülerini işleme fonksiyonu
def process_test_images(model, hr_test_dir, lr_test_dir, output_dir, grayscale=True):
    """
    Test setindeki görüntüleri işleyip sonuçları değerlendirir

    Args:
        model: Yüklenmiş SR modeli
        hr_test_dir: Yüksek çözünürlüklü test görüntüleri dizini
        lr_test_dir: Düşük çözünürlüklü test görüntüleri dizini
        output_dir: Çıktı görüntülerinin kaydedileceği dizin
        grayscale: Görüntülerin gri tonlamalı olup olmadığı

    Returns:
        psnr_scores: Tüm test görüntüleri için PSNR değerleri
        ssim_scores: Tüm test görüntüleri için SSIM değerleri
    """
    # Çıktı dizinini oluştur
    os.makedirs(output_dir, exist_ok=True)

    psnr_scores = []
    ssim_scores = []

    # Test görüntülerinin dosya adlarını al
    test_images = os.listdir(lr_test_dir)

    for img_name in tqdm(test_images):
        # Düşük ve yüksek çözünürlüklü görüntü yolları
        lr_path = os.path.join(lr_test_dir, img_name)
        hr_path = os.path.join(hr_test_dir, img_name)

        # Görüntüleri yükle
        lr_img = load_and_preprocess_image(lr_path, grayscale=grayscale)
        hr_img = load_and_preprocess_image(hr_path, grayscale=grayscale)

        # Model için giriş formatına çevir
        lr_input = np.expand_dims(lr_img, axis=0)  # Batch boyutu ekle

        # Super Resolution tahmini yap
        sr_img = model.predict(lr_input)[0]

        # PSNR ve SSIM hesapla
        # TensorFlow PSNR
        psnr_value = tf.image.psnr(hr_img, sr_img, max_val=1.0).numpy()
        psnr_scores.append(psnr_value)

        # scikit-image SSIM
        if grayscale:
            # Gri tonlamalı görüntüler için
            ssim_value = ssim(hr_img[..., 0], sr_img[..., 0], data_range=1.0)
        else:
            # Renkli görüntüler için çok kanallı SSIM
            ssim_value = ssim(hr_img, sr_img, data_range=1.0, multichannel=True)

        ssim_scores.append(ssim_value)

        # Görselleştirme için 0-255 aralığına dönüştür
        sr_display = (sr_img * 255).astype(np.uint8)
        hr_display = (hr_img * 255).astype(np.uint8)
        lr_display = (lr_img * 255).astype(np.uint8)

        # Karşılaştırma görüntüsünü oluştur
        if grayscale:
            # Karşılaştırma için üç görüntüyü yan yana koy
            comparison = np.hstack([
                cv2.resize(lr_display[..., 0], (hr_display.shape[1], hr_display.shape[0])),
                sr_display[..., 0],
                hr_display[..., 0]
            ])
            # Görüntüyü kaydet
            cv2.imwrite(os.path.join(output_dir, f"{img_name.split('.')[0]}_comparison.png"), comparison)
        else:
            # Renkli görüntüler için
            # Karşılaştırma için üç görüntüyü yan yana koy
            lr_resized = cv2.resize(lr_display, (hr_display.shape[1], hr_display.shape[0]))
            comparison = np.hstack([lr_resized, sr_display, hr_display])
            # BGR formatına çevir (OpenCV için)
            comparison_bgr = cv2.cvtColor(comparison, cv2.COLOR_RGB2BGR)
            # Görüntüyü kaydet
            cv2.imwrite(os.path.join(output_dir, f"{img_name.split('.')[0]}_comparison.png"), comparison_bgr)

        # Sonuçları ayrıca kaydet
        sr_path = os.path.join(output_dir, f"{img_name.split('.')[0]}_SR.png")
        if grayscale:
            cv2.imwrite(sr_path, sr_display[..., 0])
        else:
            cv2.imwrite(sr_path, cv2.cvtColor(sr_display, cv2.COLOR_RGB2BGR))

        # Başlık ve psnr, ssim değerleriyle görüntüleri göster
        plt.figure(figsize=(15, 5))

        if grayscale:
            plt.subplot(1, 3, 1)
            plt.title('Düşük Çözünürlük')
            plt.imshow(lr_display[..., 0], cmap='gray')

            plt.subplot(1, 3, 2)
            plt.title(f'Super Resolution\nPSNR: {psnr_value:.2f}, SSIM: {ssim_value:.4f}')
            plt.imshow(sr_display[..., 0], cmap='gray')

            plt.subplot(1, 3, 3)
            plt.title('Yüksek Çözünürlük (Ground Truth)')
            plt.imshow(hr_display[..., 0], cmap='gray')
        else:
            plt.subplot(1, 3, 1)
            plt.title('Düşük Çözünürlük')
            plt.imshow(lr_display)

            plt.subplot(1, 3, 2)
            plt.title(f'Super Resolution\nPSNR: {psnr_value:.2f}, SSIM: {ssim_value:.4f}')
            plt.imshow(sr_display)

            plt.subplot(1, 3, 3)
            plt.title('Yüksek Çözünürlük (Ground Truth)')
            plt.imshow(hr_display)

        plt.tight_layout()
        plt.savefig(os.path.join(output_dir, f"{img_name.split('.')[0]}_plot.png"))
        plt.close()

    return psnr_scores, ssim_scores


In [162]:
# Ana test fonksiyonu
def test_model():
    # Model yükleme
    # 1. Özel katman ve kayıp fonksiyonlarıyla doğrudan modeli yükle
    custom_objects = {
        'SubPixelConv2D': SubPixelConv2D,
        'psnr_loss': psnr_loss,
        'psnr_metric': psnr_metric
    }

    # Modeli yükleyin (model.h5 veya sadece ağırlıkları)
    try:
        # Tam modeli yüklemeyi deneyin
        model = load_model('/content/drive/MyDrive/srcnn_dataset/RESNET_SRCNN/final_model.h5', custom_objects=custom_objects)
        print("Tam model başarıyla yüklendi.")
    except:
        # Eğer tam model yüklenemezse, modeli yeniden oluşturup ağırlıkları yükleyin
        print("Tam model yüklenemedi, ağırlıkları yüklemeye çalışıyorum...")

        def create_resnet_srcnn(input_shape=(32, 32, 1)):
            inputs = Input(shape=input_shape)
            x = inputs

            x = Conv2D(64, (9, 9), padding='same', activation='relu')(x)
            for _ in range(10):
                skip = x
                x = Conv2D(64, (3, 3), padding='same', activation='relu')(x)
                x = Conv2D(64, (3, 3), padding='same')(x)
                x = Dropout(0.1)(x)
                x = Add()([x, skip])

            x = Conv2D(32, (5, 5), padding='same', activation='relu')(x)
            x = Conv2D(64 * 4, (3, 3), padding='same', activation='relu')(x)
            x = SubPixelConv2D(upscale_factor=2)(x)
            x = Conv2D(1, (3, 3), padding='same')(x)
            outputs = x

            model = Model(inputs=inputs, outputs=outputs)
            optimizer = Adam(learning_rate=0.0001)

            model.compile(
                optimizer=optimizer,
                loss=psnr_loss,
                metrics=[psnr_metric]
            )

            return model

        # Modeli oluştur (giriş boyutunu test verilerinize göre ayarlayın)
        model = create_resnet_srcnn(input_shape=(None, None, 1))  # Dinamik boyut için None kullanıyoruz

        # Ağırlıkları yükleyin
        try:
            model.load_weights('/content/drive/MyDrive/srcnn_dataset/RESNET_SRCNN/final_weights.weights.h5')
            print("Ağırlıklar başarıyla yüklendi.")
        except:
            model.load_weights('/content/drive/MyDrive/srcnn_dataset/RESNET_SRCNN/best_model.weights.h5')
            print("En iyi model ağırlıkları başarıyla yüklendi.")

    # Model özetini yazdır
    model.summary()

    # Test dizinlerini tanımlayın
    hr_test_dir = "/content/drive/MyDrive/srcnn_dataset/dataset_srcnn/HR/test"  # Test için yüksek çözünürlüklü görüntüler
    lr_test_dir = "/content/drive/MyDrive/srcnn_dataset/dataset_srcnn/LR/test"  # Test için düşük çözünürlüklü görüntüler
    output_dir = "/content/drive/MyDrive/srcnn_dataset/RESNET_SRCNN/results"  # Sonuçların kaydedileceği dizin

    # Test görüntülerini işle
    psnr_scores, ssim_scores = process_test_images(model, hr_test_dir, lr_test_dir, output_dir, grayscale=True)

    # Ortalama metrik sonuçlarını yazdır
    avg_psnr = np.mean(psnr_scores)
    avg_ssim = np.mean(ssim_scores)

    print(f"Ortalama PSNR: {avg_psnr:.2f} dB")
    print(f"Ortalama SSIM: {avg_ssim:.4f}")

    # Sonuçları görselleştir
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.boxplot(psnr_scores)
    plt.title(f'PSNR Değerleri\nOrtalama: {avg_psnr:.2f} dB')
    plt.grid(True, alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.boxplot(ssim_scores)
    plt.title(f'SSIM Değerleri\nOrtalama: {avg_ssim:.4f}')
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'metrics_results.png'))
    plt.show()

    # Detaylı sonuçları bir dosyaya kaydet
    with open(os.path.join(output_dir, 'test_results.txt'), 'w') as f:
        f.write(f"Ortalama PSNR: {avg_psnr:.2f} dB\n")
        f.write(f"Ortalama SSIM: {avg_ssim:.4f}\n\n")
        f.write("Görüntü bazlı sonuçlar:\n")

        test_images = os.listdir(lr_test_dir)
        for i, img_name in enumerate(test_images):
            f.write(f"{img_name}: PSNR = {psnr_scores[i]:.2f} dB, SSIM = {ssim_scores[i]:.4f}\n")


In [163]:
# Kodu çalıştır
if __name__ == "__main__":
    test_model()

Tam model başarıyla yüklendi.


Model: "functional_18"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_15            │ (None, 32, 32, 1)      │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_248 (Conv2D)       │ (None, 32, 32, 64)     │          5,248 │ input_layer_15[0][0]   │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_249 (Conv2D)       │ (None, 32, 32, 64)     │         36,928 │ conv2d_248[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_250 (Conv2D)       │ (None, 32, 32, 64)     │         36,928 │ conv2d_249[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_70 (Dropout)      │ (None, 32, 32, 64)     │              0 │ conv2d_250[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_100 (Add)             │ (None, 32, 32, 64)     │              0 │ dropout_70[0][0],      │
│                           │                        │                │ conv2d_248[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_251 (Conv2D)       │ (None, 32, 32, 64)     │         36,928 │ add_100[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_252 (Conv2D)       │ (None, 32, 32, 64)     │         36,928 │ conv2d_251[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_71 (Dropout)      │ (None, 32, 32, 64)     │              0 │ conv2d_252[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_101 (Add)             │ (None, 32, 32, 64)     │              0 │ dropout_71[0][0],      │
│                           │                        │                │ add_100[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_253 (Conv2D)       │ (None, 32, 32, 64)     │         36,928 │ add_101[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_254 (Conv2D)       │ (None, 32, 32, 64)     │         36,928 │ conv2d_253[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_72 (Dropout)      │ (None, 32, 32, 64)     │              0 │ conv2d_254[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_102 (Add)             │ (None, 32, 32, 64)     │              0 │ dropout_72[0][0],      │
│                           │                        │                │ add_101[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_255 (Conv2D)       │ (None, 32, 32, 64)     │         36,928 │ add_102[0][0]          │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_256 (Conv2D)       │ (None, 32, 32, 64)     │         36,928 │ conv2d_255[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout_73 (Dropout)      │ (None, 32, 32, 64)     │              0 │ conv2d_256[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_103 (Add)        

 Total params: 869,603 (3.32 MB)

 Trainable params: 869,601 (3.32 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

  0%|          | 0/414 [00:00<?, ?it/s]


ValueError: Input 0 of layer "functional_18" is incompatible with the layer: expected shape=(None, 32, 32, 1), found shape=(1, 270, 512)

In [ ]:
# TEST
if __name__ == "__main__":
  model = create_resnet_srcnn()
  model.summary()

  #Eğitilmiş modelin ağırlıklarını yükle
  try:
    model.load.weights('')
    print("Ağırlıklar başarıyla yüklendi.")
  except:
    try:
      model.load_weights('')
      print("En iyi model ağırlıkları başarıyla yüklendi.")
    except Exception as e:
      print(f"Ağırlıklar yüklenemedi: {e}")

    # Artık model herhangi bir boyuttaki görüntüyü işleyebilir
    print("Model herhangi bir boyuttaki görüntüyü işleyebilir!")